# Calibrate ZSCORE_DIVISOR Constants

Finds the `ZSCORE_DIVISOR` values in `matchup_predictor.py` that minimize log-loss against observed
`plate_appearances` outcomes. Results should be written back to `config/multi_elo_config.yaml`
under `prediction_engine.zscore_divisors`.

## Method
1. Pull a sample of PAs from Supabase with their result types.
2. Join with `talent_player_current` to get batter/pitcher ELO at time of season (approximation;
   for per-PA ELO use `talent_daily_ohlc` join on game_date).
3. Compute predicted outcome probabilities using the 3-stage model.
4. Minimize multi-class log-loss using `scipy.optimize.minimize`.
5. Report optimal divisors and log-loss improvement vs baseline.

In [1]:
import os
import math
import yaml
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from supabase import create_client
from dotenv import load_dotenv

load_dotenv()
supabase = create_client(os.environ['SUPABASE_URL'], os.environ['SUPABASE_KEY'])
print('Connected to Supabase')

Connected to Supabase


In [2]:
# ---------------------------------------------------------------------------
# 1. Pull plate appearances with result types
# ---------------------------------------------------------------------------
SAMPLE_LIMIT = 50_000  # increase for more accurate calibration
PAGE_SIZE = 1_000      # Supabase PostgREST max rows per request

# Outcome categories we model
OUTCOME_MAP = {
    'BB': 'BB', 'IBB': 'BB', 'HBP': 'HBP',
    'StrikeOut': 'K',
    'Single': '1B', 'Double': '2B', 'Triple': '3B', 'HR': 'HR',
    'OUT': 'OUT', 'FC': 'OUT', 'GIDP': 'OUT',
    'POPUP': 'OUT', 'GROUNDOUT': 'OUT', 'SAC': 'OUT', 'E': 'OUT',
}
OUTCOMES = ['BB', 'HBP', 'K', 'OUT', '1B', '2B', '3B', 'HR']

# Paginate â€” Supabase caps single responses at 1,000 rows
all_rows = []
offset = 0
while len(all_rows) < SAMPLE_LIMIT:
    resp = (
        supabase.table('plate_appearances')
        .select('pa_id, batter_id, pitcher_id, result_type, game_date')
        .not_.is_('batter_id', 'null')
        .not_.is_('pitcher_id', 'null')
        .order('game_date', desc=False)
        .range(offset, offset + PAGE_SIZE - 1)
        .execute()
    )
    page = resp.data or []
    if not page:
        break
    all_rows.extend(page)
    offset += len(page)
    if len(page) < PAGE_SIZE:
        break

pa_df = pd.DataFrame(all_rows[:SAMPLE_LIMIT])
pa_df['outcome'] = pa_df['result_type'].map(OUTCOME_MAP)
pa_df = pa_df.dropna(subset=['outcome'])
pa_df['game_date'] = pd.to_datetime(pa_df['game_date']).dt.date
print(f'Loaded {len(pa_df):,} plate appearances')
print(f'Date range: {pa_df["game_date"].min()} â†’ {pa_df["game_date"].max()}')
pa_df['outcome'].value_counts(normalize=True).round(4)

Loaded 50,000 plate appearances
Date range: 2025-03-27 â†’ 2025-05-16


outcome
OUT    0.4656
K      0.2196
1B     0.1438
BB     0.0874
2B     0.0411
HR     0.0287
HBP    0.0104
3B     0.0034
Name: proportion, dtype: float64

In [3]:
# ---------------------------------------------------------------------------
# 2. Load batter and pitcher ELO (season snapshot from talent_player_current)
# ---------------------------------------------------------------------------
all_ids = list(set(pa_df['batter_id'].tolist() + pa_df['pitcher_id'].tolist()))

elo_rows = []
batch_size = 200
for i in range(0, len(all_ids), batch_size):
    batch = all_ids[i:i + batch_size]
    resp = (
        supabase.table('talent_player_current')
        .select('player_id, player_role, talent_type, season_elo')
        .in_('player_id', batch)
        .execute()
    )
    elo_rows.extend(resp.data or [])

elo_df = pd.DataFrame(elo_rows)
elo_pivot = elo_df.pivot_table(
    index=['player_id', 'player_role'], columns='talent_type', values='season_elo'
).reset_index()

batter_elo = elo_pivot[elo_pivot['player_role'] == 'batter'].set_index('player_id')
pitcher_elo = elo_pivot[elo_pivot['player_role'] == 'pitcher'].set_index('player_id')
print(f'Batter ELO: {len(batter_elo):,} players | Pitcher ELO: {len(pitcher_elo):,} players')

Batter ELO: 596 players | Pitcher ELO: 631 players


In [4]:
# ---------------------------------------------------------------------------
# 3. Build feature matrix: z-score diffs for each PA
# ---------------------------------------------------------------------------
# ELO distribution constants (from matchup_predictor.py)
DIST = {
    'contact':         {'mean': 1504.5, 'std': 33.9},
    'power':           {'mean': 1468.6, 'std': 61.6},
    'discipline':      {'mean': 1700.3, 'std': 139.0},
    'stuff':           {'mean': 1587.3, 'std': 56.6},
    'bip_suppression': {'mean': 1513.3, 'std': 18.2},
    'command':         {'mean': 1681.1, 'std': 126.5},
}

def zscore(val, dim):
    d = DIST[dim]
    return (val - d['mean']) / d['std'] if d['std'] > 0 else 0.0

DEFAULT_BATTER = {'contact': 1504.5, 'power': 1468.6, 'discipline': 1700.3}
DEFAULT_PITCHER = {'stuff': 1587.3, 'bip_suppression': 1513.3, 'command': 1681.1}

def get_batter_elos(pid):
    if pid in batter_elo.index:
        row = batter_elo.loc[pid]
        return {
            'contact': row.get('contact', DEFAULT_BATTER['contact']) or DEFAULT_BATTER['contact'],
            'power': row.get('power', DEFAULT_BATTER['power']) or DEFAULT_BATTER['power'],
            'discipline': row.get('discipline', DEFAULT_BATTER['discipline']) or DEFAULT_BATTER['discipline'],
        }
    return DEFAULT_BATTER

def get_pitcher_elos(pid):
    if pid in pitcher_elo.index:
        row = pitcher_elo.loc[pid]
        return {
            'stuff': row.get('stuff', DEFAULT_PITCHER['stuff']) or DEFAULT_PITCHER['stuff'],
            'bip_suppression': row.get('bip_suppression', DEFAULT_PITCHER['bip_suppression']) or DEFAULT_PITCHER['bip_suppression'],
            'command': row.get('command', DEFAULT_PITCHER['command']) or DEFAULT_PITCHER['command'],
        }
    return DEFAULT_PITCHER

features = []
for _, row in pa_df.iterrows():
    b = get_batter_elos(row['batter_id'])
    p = get_pitcher_elos(row['pitcher_id'])
    features.append({
        'z_disc_cmd': zscore(b['discipline'], 'discipline') - zscore(p['command'], 'command'),
        'z_stuff_contact': zscore(p['stuff'], 'stuff') - zscore(b['contact'], 'contact'),
        'z_contact_bip': zscore(b['contact'], 'contact') - zscore(p['bip_suppression'], 'bip_suppression'),
        'z_stuff_power': zscore(p['stuff'], 'stuff') - zscore(b['power'], 'power'),
        'outcome': row['outcome'],
        'game_date': row['game_date'],
    })

feat_df = pd.DataFrame(features)
print('Feature matrix shape:', feat_df.shape)
feat_df.head(3)

# Temporal 80/20 train/test split â€” prevents overfitting to full dataset
feat_df = feat_df.sort_values('game_date').reset_index(drop=True)
split_idx = int(len(feat_df) * 0.80)
train_df = feat_df.iloc[:split_idx].copy()
test_df  = feat_df.iloc[split_idx:].copy()
print(f'Feature matrix: {len(feat_df):,} rows')
print(f'Train: {len(train_df):,} PAs ({train_df["game_date"].min()} â†’ {train_df["game_date"].max()})')
print(f'Test:  {len(test_df):,} PAs  ({test_df["game_date"].min()} â†’ {test_df["game_date"].max()})')
feat_df.head(3)

Feature matrix shape: (50000, 6)
Feature matrix: 50,000 rows
Train: 40,000 PAs (2025-03-27 â†’ 2025-05-06)
Test:  10,000 PAs  (2025-05-06 â†’ 2025-05-16)


,z_disc_cmd,z_stuff_contact,z_contact_bip,z_stuff_power,outcome,game_date
0,-0.616470,0.117890,0.375985,1.976317,OUT,2025-03-27
1,-0.023049,1.762834,-1.083904,1.752694,K,2025-03-27
2,0.364361,-1.999997,2.678928,-0.452013,K,2025-03-27


In [5]:
# ---------------------------------------------------------------------------
# 4. Prediction function parameterized by divisors
# ---------------------------------------------------------------------------
LA = {
    'bb_rate': 0.0949, 'k_rate': 0.2218, 'bip_rate': 0.6834,
    'hit_rate_on_bip': 0.3206, 'xbh_rate_on_hit': 0.3493,
    '2b_ratio': 0.5522, '3b_ratio': 0.0448, 'hr_ratio': 0.403,
}
HBP_FRAC = 0.011 / LA['bb_rate']

def predict_probs(z_disc_cmd, z_stuff_contact, z_contact_bip, z_stuff_power, divisors):
    d_bb, d_k, d2, d3 = divisors

    # Stage 1
    bip = LA['bip_rate']
    logit_bb = math.log(LA['bb_rate'] / bip) + z_disc_cmd / d_bb
    logit_k  = math.log(LA['k_rate']  / bip) + z_stuff_contact / d_k
    exp_bb, exp_k = math.exp(logit_bb), math.exp(logit_k)
    denom = exp_bb + exp_k + 1.0
    p_bb_bucket = exp_bb / denom
    p_k  = exp_k  / denom
    p_bip = 1.0   / denom

    # LR-3: split BB into BB + HBP
    p_hbp = p_bb_bucket * HBP_FRAC
    p_bb  = p_bb_bucket * (1.0 - HBP_FRAC)

    # Stage 2
    base_hit_logit = math.log(LA['hit_rate_on_bip'] / (1.0 - LA['hit_rate_on_bip']))
    p_hit_given_bip = 1.0 / (1.0 + math.exp(-(base_hit_logit + z_contact_bip / d2)))
    p_hit = p_bip * p_hit_given_bip
    p_out = p_bip * (1.0 - p_hit_given_bip)

    # Stage 3
    base_xbh_logit = math.log(LA['xbh_rate_on_hit'] / (1.0 - LA['xbh_rate_on_hit']))
    p_xbh_given_hit = 1.0 / (1.0 + math.exp(-(base_xbh_logit + (-z_stuff_power) / d3)))
    p_1b = p_hit * (1.0 - p_xbh_given_hit)
    p_xbh = p_hit * p_xbh_given_hit

    return {
        'BB': max(1e-9, p_bb), 'HBP': max(1e-9, p_hbp),
        'K': max(1e-9, p_k),   'OUT': max(1e-9, p_out),
        '1B': max(1e-9, p_1b),
        '2B': max(1e-9, p_xbh * LA['2b_ratio']),
        '3B': max(1e-9, p_xbh * LA['3b_ratio']),
        'HR': max(1e-9, p_xbh * LA['hr_ratio']),
    }


def log_loss(divisors, df):
    """Mean multi-class log-loss across all PAs."""
    total = 0.0
    for _, row in df.iterrows():
        probs = predict_probs(
            row['z_disc_cmd'], row['z_stuff_contact'],
            row['z_contact_bip'], row['z_stuff_power'],
            divisors,
        )
        p = probs.get(row['outcome'], 1e-9)
        total -= math.log(max(1e-9, p))
    return total / len(df)


# Baseline loss with current divisors
CURRENT = [3.7907, 8.3045, 20.0, 15.4005]  # current config values from multi_elo_config.yaml
baseline_loss = log_loss(CURRENT, feat_df)
print(f'Baseline log-loss: {baseline_loss:.5f}')

Baseline log-loss: 1.47532


In [6]:
# ---------------------------------------------------------------------------
# 5b. Grid search: find optimal single flat divisor on TRAIN set
#     This becomes the baseline for cell-6 and the fallback for cell-8.
# ---------------------------------------------------------------------------
grid_values = np.arange(2.0, 20.5, 0.5)
grid_results = []
for d in grid_values:
    loss = log_loss([d, d, d, d], train_df)
    grid_results.append({'divisor': d, 'train_loss': loss})

grid_df = pd.DataFrame(grid_results)
best_row = grid_df.loc[grid_df['train_loss'].idxmin()]
BEST_FLAT = float(best_row['divisor'])
flat_test_loss = log_loss([BEST_FLAT] * 4, test_df)

print(grid_df.to_string(index=False))
print()
print(f'Best single flat divisor (train): {BEST_FLAT}')
print(f'Held-out log-loss (best flat):    {flat_test_loss:.5f}')

 divisor  train_loss
     2.0    1.575382
     2.5    1.535509
     3.0    1.514203
     3.5    1.501743
     4.0    1.493972
     4.5    1.488888
     5.0    1.485439
     5.5    1.483032
     6.0    1.481315
     6.5    1.480069
     7.0    1.479154
     7.5    1.478475
     8.0    1.477970
     8.5    1.477592
     9.0    1.477310
     9.5    1.477101
    10.0    1.476948
    10.5    1.476838
    11.0    1.476762
    11.5    1.476712
    12.0    1.476683
    12.5    1.476671
    13.0    1.476671
    13.5    1.476682
    14.0    1.476700
    14.5    1.476726
    15.0    1.476756
    15.5    1.476790
    16.0    1.476827
    16.5    1.476866
    17.0    1.476907
    17.5    1.476949
    18.0    1.476993
    18.5    1.477036
    19.0    1.477080
    19.5    1.477124
    20.0    1.477168

Best single flat divisor (train): 12.5
Held-out log-loss (best flat):    1.47728


In [7]:
# ---------------------------------------------------------------------------
# 5. Optimize divisors on train set; evaluate on held-out test set
# ---------------------------------------------------------------------------
result = minimize(
    log_loss,
    x0=CURRENT,
    args=(train_df,),
    method='L-BFGS-B',
    bounds=[(1.0, 30.0)] * 4,
    options={'maxiter': 200, 'ftol': 1e-8},
)

opt = result.x.tolist()

# Evaluate on held-out test set to check generalization
held_out_loss = log_loss(opt, test_df)
flat_loss     = log_loss([BEST_FLAT] * 4, test_df)
baseline_loss = log_loss(CURRENT, train_df)

print(f'Optimization success: {result.success}')
print(f'Optimal divisors:  stage1_bb={opt[0]:.3f}  stage1_k={opt[1]:.3f}  stage2={opt[2]:.3f}  stage3={opt[3]:.3f}')
print()
print(f'Train log-loss (optimized): {result.fun:.5f}  (vs baseline {baseline_loss:.5f})')
print(f'Test  log-loss (optimized): {held_out_loss:.5f}')
print(f'Test  log-loss (flat {BEST_FLAT}):  {flat_loss:.5f}')
print()
if held_out_loss < flat_loss:
    print(f'PASS: optimized divisors beat best flat ({BEST_FLAT}) on held-out data')
else:
    print('FAIL: optimized divisors overfit â€” held-out loss worse than flat baseline')
    print('      Keeping current config; domain-motivated fixes (stage1_bb) still apply.')

Optimization success: True
Optimal divisors:  stage1_bb=3.791  stage1_k=8.304  stage2=20.000  stage3=15.401

Train log-loss (optimized): 1.47535  (vs baseline 1.47535)
Test  log-loss (optimized): 1.47521
Test  log-loss (flat 12.5):  1.47728

PASS: optimized divisors beat best flat (12.5) on held-out data


In [8]:
# ---------------------------------------------------------------------------
# 6. Per-outcome Brier score comparison
# ---------------------------------------------------------------------------
def brier_scores(divisors, df):
    scores = {o: [] for o in OUTCOMES}
    for _, row in df.iterrows():
        probs = predict_probs(
            row['z_disc_cmd'], row['z_stuff_contact'],
            row['z_contact_bip'], row['z_stuff_power'],
            divisors,
        )
        for outcome in OUTCOMES:
            p_pred = probs.get(outcome, 0.0)
            actual = 1.0 if row['outcome'] == outcome else 0.0
            scores[outcome].append((p_pred - actual) ** 2)
    return {o: np.mean(v) for o, v in scores.items()}

base_brier = brier_scores(CURRENT, feat_df)
opt_brier  = brier_scores(opt, feat_df)

comparison = pd.DataFrame({'baseline': base_brier, 'optimized': opt_brier})
comparison['delta'] = comparison['optimized'] - comparison['baseline']
print('\nBrier score per outcome (lower = better):')
print(comparison.round(6))


Brier score per outcome (lower = better):
     baseline  optimized  delta
BB   0.079634   0.079634   -0.0
HBP  0.010272   0.010272    0.0
K    0.170095   0.170095   -0.0
OUT  0.248378   0.248378   -0.0
1B   0.122954   0.122954    0.0
2B   0.039375   0.039375   -0.0
3B   0.003388   0.003388    0.0
HR   0.027878   0.027878    0.0


In [9]:
# ---------------------------------------------------------------------------
# 7. Write divisors to config
#    - If 4-param optimization beats best flat on held-out: write per-stage values.
#    - Otherwise: write best flat divisor (optimal 1-param, no overfitting).
# ---------------------------------------------------------------------------
CONFIG_PATH = '../config/multi_elo_config.yaml'

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

if held_out_loss < flat_loss:
    new_divisors = {
        'stage1_bb':  round(opt[0], 4),
        'stage1_k':   round(opt[1], 4),
        'stage2':     round(opt[2], 4),
        'stage3':     round(opt[3], 4),
        'stage1_hbp': cfg['prediction_engine']['zscore_divisors'].get('stage1_hbp', 7.0),
    }
    print('Using per-stage optimized divisors (beat flat on held-out data).')
else:
    new_divisors = {
        'stage1_bb':  BEST_FLAT,
        'stage1_k':   BEST_FLAT,
        'stage2':     BEST_FLAT,
        'stage3':     BEST_FLAT,
        'stage1_hbp': cfg['prediction_engine']['zscore_divisors'].get('stage1_hbp', 7.0),
    }
    print(f'4-param fit overfit -- using best flat divisor ({BEST_FLAT}) for all stages.')

print('Proposed zscore_divisors:')
for k, v in new_divisors.items():
    old = cfg['prediction_engine']['zscore_divisors'].get(k, 'N/A')
    print(f'  {k}: {old} -> {v}')

cfg['prediction_engine']['zscore_divisors'] = new_divisors
with open(CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)
print('Config updated.')

Using per-stage optimized divisors (beat flat on held-out data).
Proposed zscore_divisors:
  stage1_bb: 3.7907 -> 3.7906
  stage1_k: 8.3045 -> 8.3045
  stage2: 20.0 -> 20.0
  stage3: 15.4005 -> 15.4005
  stage1_hbp: 7.0 -> 7.0
Config updated.
